# Анализ данных домена Retail за последние 30 дней

Retail описывает сервис доставки продуктов. Здесь фиксируется полный путь пользователя — от просмотра до оформления заказа, включая добавление в корзину. Контексты включают поиск, каталог, страницы товаров и корзину. https://habr.com/ru/companies/tbank/articles/950696/

In [2]:
import os
import pandas as pd
import numpy as np
import glob
#from matplotlib import pyplot as plt

In [3]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/retail/events"

In [4]:
# Получаем список всех файлов .pq
event_files = [f for f in os.listdir(events_dir) if f.endswith('.pq')]

In [5]:
# Отсортируем по номеру дня
event_files_sorted = sorted(event_files, reverse=True)

In [6]:
# Возьмем последние 30 файлов
last_30_files = event_files_sorted[:30]

In [7]:
# Считываем и объединяем данные
dfs = []
for fname in last_30_files:
    filepath = os.path.join(events_dir, fname)
    df = pd.read_parquet(filepath)
    dfs.append(df)

In [8]:
# Объединенный DataFrame за последние 30 дней
recent_events = pd.concat(dfs, ignore_index=True)

In [9]:
users_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/users.pq')
items_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/retail/items.pq')

In [10]:
brands_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/brands.pq', engine='fastparquet')

# Исследуем таблицу recent_events

In [11]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200006078,8923243,fmcg_208146,catalog,view,android
1,113011200032272,12124655,fmcg_1034096,catalog,view,android
2,113011200077336,20658160,fmcg_112697,catalog,view,android
3,113011200112253,85296647,fmcg_618508,catalog,view,android
4,113011200250868,85296647,fmcg_509248,catalog,view,android


In [12]:
recent_events.shape

(47376689, 6)

In [13]:
recent_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47376689 entries, 0 to 47376688
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   timestamp    int64 
 1   user_id      uint64
 2   item_id      object
 3   subdomain    object
 4   action_type  object
 5   os           object
dtypes: int64(1), object(4), uint64(1)
memory usage: 2.1+ GB


In [14]:
print(f'Уникальных пользователей: {recent_events.user_id.nunique()}')
print(f'Уникальных айтемов: {recent_events.item_id.nunique()}')

Уникальных пользователей: 33885
Уникальных айтемов: 185416


#### Распределение операционных систем

In [17]:
recent_events['os'].unique()

array(['android', 'ios'], dtype=object)

In [18]:
recent_events.groupby('os', dropna=False)['user_id'].count().sort_values(ascending=False)

os
android    42845021
ios         4531668
Name: user_id, dtype: int64

Из таблицы видно, что чаще всего используемая ОС - android

In [19]:
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

#### Распределение типов действий

In [20]:
recent_events.groupby('action_type', dropna=False)['user_id'].count().sort_values(ascending=False)

action_type
view             46275844
added-to-cart      697232
click              403613
Name: user_id, dtype: int64

Самое частое действие - просмотр. Следующее - добавление в корзину. И менее всего кликов.

#### Распределение событий по контекстам, где они происходили (каталог, поиск, добавление в корзину, страницу товара)

main - страница товара? item - похожий товар?

In [21]:
recent_events['subdomain'].unique()

array(['catalog', 'search', 'item', 'main', 'cart'], dtype=object)

In [22]:
recent_events.groupby('subdomain', dropna=False)['user_id'].count().sort_values(ascending=False)

subdomain
catalog    41782814
search      4260983
main         637259
item         485620
cart         210013
Name: user_id, dtype: int64

Из таблицы видно, что наибольшее количество взаимодействий пользователей с товаром произошло в каталоге.

### Работа со временем. Предположение - время задано в микросекундах со сдвигом

In [23]:
recent_events['timestamp'].min(), recent_events['timestamp'].max()

(np.int64(110505600584983), np.int64(113097599976424))

In [24]:
# конвертируем в datetime (получим примерно 1973 год)
recent_events['datetime_wrong_year'] = pd.to_datetime(recent_events['timestamp'], unit='us')

In [25]:
# желаемая и фактическая даты начала
# .normalize() убирает время, оставляя только дату
desired_start_date = pd.to_datetime('2025-01-01')
actual_start_date = recent_events['datetime_wrong_year'].min().normalize() 

In [26]:
# сдвиг
time_offset = desired_start_date - actual_start_date

In [27]:
# применяем сдвиг ко всем датам
recent_events['datetime_corrected'] = recent_events['datetime_wrong_year'] + time_offset

In [28]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').head())

                timestamp        datetime_wrong_year  \
45982150  110505600584983 1973-07-03 00:00:00.584983   
45982151  110505600711834 1973-07-03 00:00:00.711834   
45982152  110505601068406 1973-07-03 00:00:01.068406   
45982153  110505601223658 1973-07-03 00:00:01.223658   
45982154  110505601354955 1973-07-03 00:00:01.354955   

                 datetime_corrected  
45982150 2025-01-01 00:00:00.584983  
45982151 2025-01-01 00:00:00.711834  
45982152 2025-01-01 00:00:01.068406  
45982153 2025-01-01 00:00:01.223658  
45982154 2025-01-01 00:00:01.354955  


In [29]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').tail())

               timestamp        datetime_wrong_year         datetime_corrected
1238049  113097599527555 1973-08-01 23:59:59.527555 2025-01-30 23:59:59.527555
1238050  113097599558412 1973-08-01 23:59:59.558412 2025-01-30 23:59:59.558412
1238051  113097599723432 1973-08-01 23:59:59.723432 2025-01-30 23:59:59.723432
1238052  113097599841644 1973-08-01 23:59:59.841644 2025-01-30 23:59:59.841644
1238053  113097599976424 1973-08-01 23:59:59.976424 2025-01-30 23:59:59.976424


In [30]:
print(recent_events['datetime_corrected'].max() - recent_events['datetime_corrected'].min())

29 days 23:59:59.391441


# Исследуем данные из таблицы users_df  (таблица users представлена и согласована во всех доменах)

In [21]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21.0,2.0
1,72517894,10.0,90.0
2,86699708,9.0,9.0
3,54241043,17.0,58.0
4,23591057,17.0,4.0


In [22]:
users_df.shape

(3500000, 3)

In [23]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500000 entries, 0 to 3499999
Data columns (total 3 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user_id         uint64 
 1   socdem_cluster  float64
 2   region          float64
dtypes: float64(2), uint64(1)
memory usage: 80.1 MB


In [24]:
users_df['socdem_cluster'].nunique()

21

In [25]:
users_df['region'].nunique()

90

#### Распределние по социо-демографическим кластерам (первые пять наиболее заполненнных)

In [26]:
users_df.groupby('socdem_cluster', dropna=False)['user_id'].count().sort_values(ascending=False).head()

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
Name: user_id, dtype: int64

#### Распределение по регионам

In [27]:
users_df.groupby('region', dropna=False)['user_id'].count().sort_values(ascending=False).head()

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
Name: user_id, dtype: int64

# Исследуем данные из таблицы items_df

In [28]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,fmcg_10,65693,Office Supplies and Educational Materials,Stationery Paper,-4.406018,"[-0.13557312, 0.049446248, -0.018282967, 0.025..."
1,fmcg_10000,146468,Cleaning Supplies and Everyday Household Items,Cleaning and Detergent Products,-4.205185,"[-0.043275375, -0.035317067, 0.0033219394, -0...."
2,fmcg_1000006,37799,None,None,-4.463660,"[-0.083411045, 0.049153276, -0.08736873, 0.075..."
3,fmcg_1000008,240838,Children's Products and Childcare Items,Food Products,-5.138377,"[-0.055184077, 0.02342301, 0.03789554, 0.11559..."
4,fmcg_1000017,240838,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-3.980383,"[-0.052520722, -0.0063896044, -0.011138491, 0...."


In [29]:
items_df.shape

(250171, 6)

In [30]:
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250171 entries, 0 to 250170
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   item_id      250171 non-null  object 
 1   brand_id     250171 non-null  uint64 
 2   category     240586 non-null  object 
 3   subcategory  240586 non-null  object 
 4   price        223682 non-null  float64
 5   embedding    250171 non-null  object 
dtypes: float64(1), object(4), uint64(1)
memory usage: 11.5+ MB


In [31]:
items_df['category'].unique()

array(['Office Supplies and Educational Materials',
       'Cleaning Supplies and Everyday Household Items', None,
       "Children's Products and Childcare Items",
       'Foodstuffs and Beverages',
       'Automotive Accessories, Spare Parts, and Car Care Products',
       'Cosmetics, Personal Care, and Health Maintenance Products',
       'Pharmaceuticals and Medical Supplies',
       'Home Improvement and Countryside Retreat Essentials', 'Other',
       'Pet Supplies: Food, Accessories, and Grooming Products',
       'Outerwear, Casual Apparel, and Specialized Workwear',
       'Sports Equipment, Gear, and Outdoor Activity Accessories',
       'Hobby, Creative, and Leisure Products',
       'Construction and Renovation Materials, Tools, and Equipment',
       'Fashion Accessories, Tech Add-ons, and Style Enhancements',
       'Household Electrical Appliances',
       'Printed Media: Publications, Textbooks, and Fiction',
       'Electronic Devices and Gadgets', 'Footwear of All Typ

In [32]:
items_df.groupby('category', dropna=False)['item_id'].count().sort_values(ascending=False)

category
Foodstuffs and Beverages                                       160128
Cosmetics, Personal Care, and Health Maintenance Products       22998
Home Improvement and Countryside Retreat Essentials             17738
NaN                                                              9585
Children's Products and Childcare Items                          8339
Cleaning Supplies and Everyday Household Items                   7731
Outerwear, Casual Apparel, and Specialized Workwear              4162
Pet Supplies: Food, Accessories, and Grooming Products           3710
Other                                                            3155
Office Supplies and Educational Materials                        2983
Fashion Accessories, Tech Add-ons, and Style Enhancements        1810
Sports Equipment, Gear, and Outdoor Activity Accessories         1289
Hobby, Creative, and Leisure Products                            1252
Automotive Accessories, Spare Parts, and Car Care Products       1215
Pharmaceuti

In [33]:
items_df['subcategory'].nunique()

174

174 уникальные подкатегории

In [34]:
items_df.groupby('subcategory', dropna=False)['item_id'].count().sort_values(ascending=False)

subcategory
Dairy Products and Eggs                   22544
Bakery Products                           21650
Sweet Desserts and Confectionery          21644
Meat and Sausage Products                 14334
Beverages and Drinks                      10893
                                          ...  
Remote-Controlled Toys and 3D Modeling        1
Built-In Household Appliances                 1
Esoteric Practices and Divination             1
Wrist, Wall, and Other Watches                1
Business Accessories                          1
Name: item_id, Length: 175, dtype: int64

# Посмотрим на данные из таблицы brands_df (таблица brands представлена и согласована во всех доменах)

In [35]:
brands_df.head()

,brand_id,embedding
0,4,None
1,34,None
2,45,None
3,46,None
4,51,None


In [36]:
brands_df.shape

(24513, 2)

In [37]:
brands_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24513 entries, 0 to 24512
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   brand_id   24513 non-null  uint64
 1   embedding  6559 non-null   object
dtypes: object(1), uint64(1)
memory usage: 383.1+ KB


# Анализ пропусков 

### Пропущенные значения в датасете users_df (совпадает во всех доменах)

In [38]:
users_df.isna().sum()

user_id               0
socdem_cluster     5153
region            58917
dtype: int64

In [39]:
def missing_value_rate(df, column):
  print(f'Доля пропущенных значений в колонке {column}:  {(df[column].isna().sum() / users_df.shape[0]):.4f}')

In [40]:
for column in users_df.columns:
  missing_value_rate(users_df, column)

Доля пропущенных значений в колонке user_id:  0.0000
Доля пропущенных значений в колонке socdem_cluster:  0.0015
Доля пропущенных значений в колонке region:  0.0168


Возможные причины возникновения пропусков:
Необязательные поля при регистрации
Система не смогла определить местоположение пользователя, возможно пользователь не дал разрешение на определение местоположения
Технические ошибки при сборе данных

In [41]:
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0])
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0] / users_df.shape[0])

4667
0.0013334285714285713


Итого у нас есть 4667 записей пользователей, где пропуски в обеих колонках socdem_cluster и region. Это около 1,3%.

In [42]:
# проверим дубликаты
users_df.duplicated().sum()

np.int64(0)

In [43]:
print(f'Минимальное значение в колонках socdem_cluster и region соответвенно: {users_df['socdem_cluster'].min()} и {users_df['region'].min()}')

Минимальное значение в колонках socdem_cluster и region соответвенно: 0.0 и 0.0


Стратегия работы с пропусками

Процент пропусков очень мал. Технически можно удалить строки с пропущенными значениями в таблице users_df. Но тогда бы должны будем удалить этих пользователей из таблицы events_df, которую мы будем использовать для посторения матрицы взаимодействий и уже на этой основе строить прогноз. В таблице events_df нет пропусков.

Колонки socdem_cluster и region это категориальные данные, закодированные числами. Нам нужно заполнить пропущенные значения специальным зарезервированным числом, которое не встречается среди реальных кодов регионов и кластеров. Таким образом, мы сохраним всю историю взаимодействий c этими айтемами в events_df. Можно использовать число -1, так как минимальное значение в указанных колонках равно ноль.

# Пропущенные значения в items_df

In [44]:
items_df.isna().sum()

item_id            0
brand_id           0
category        9585
subcategory     9585
price          26489
embedding          0
dtype: int64

In [45]:
for column in items_df.columns:
  missing_value_rate(items_df, column)

Доля пропущенных значений в колонке item_id:  0.0000
Доля пропущенных значений в колонке brand_id:  0.0000
Доля пропущенных значений в колонке category:  0.0027
Доля пропущенных значений в колонке subcategory:  0.0027
Доля пропущенных значений в колонке price:  0.0076
Доля пропущенных значений в колонке embedding:  0.0000


### Возможные причины возникновения пропусков

Для category и subcategory - товар могли добавить в базу данных, но еще не успели присвоить ему категорию.

Некоторые товары могут не подходить ни под одну из существующих категорий, и поле остается пустым.

Поставшики прислали неполную информацию.

Для price - это могут быть товары не для продажи, а, например, промо-материалы.

Некоторые товары могут не иметь публичной цены, а она узнается по запросу.

Часть комплекта: товар может продаваться только в составе набора, и у него нет индивидуальной цены.

### Стратегия работы с пропусками

* в колонках `category` и `subcategory` (категориальные данные) заменить на 'unknown'.Таким образом, мы сохраняем всю его ценную историю взаимодействий в `events_df`.
* Это лучше, чем заполнять самой частой категорией, так как это может внести смещение в данные.​
* Колонка `price`. Можно заполнить нулем, если мы можем предположить, что товар, например, акционный, бесплатный. Но это частный случай и у нас нет данных для выдвижения такого предположения. Лучше заполнить медианой. Медиана более устойчива к выбросам, поэтому она лучше отражает "типичную" цену.

# Пропущенные значения в events_df

In [46]:
recent_events.isna().sum()

timestamp      0
user_id        0
item_id        0
subdomain      0
action_type    0
os             0
dtype: int64

In [47]:
recent_events['subdomain'].unique()

array(['catalog', 'search', 'item', 'main', 'cart'], dtype=object)

пропущенных значений в events нет.

# Пропущенные значения в brands_df (совпадает во всех доменах)

In [48]:
brands_df.isna().sum()

brand_id         0
embedding    17954
dtype: int64

In [49]:
print(f'Доля пропущенных значений в колонке embedding:  {(brands_df['embedding'].isna().sum() / brands_df.shape[0]):.4f}')

Доля пропущенных значений в колонке embedding:  0.7324


Более 73% брендов не имеют эмбеддинга.

* **Проблема холодного старта.** Эмбеддинги брендов обычно строятся на основе поведения пользователей (просмотры, покупки, клики). Если бренд новый или с ним было очень мало взаимодействий, у системы недостаточно данных, чтобы построить для него осмысленный эмбеддинг. Эти пропущенные эмбеддинги - это новые или непопулярные бренды.

* Эмбеддинги могут пересчитывать не в реальном времени, а, например, раз в неделю. Пропуски могут соответствовать брендам, которые были добавлены после последнего обновления и еще ждут своей очереди на обработку.

**Заметка про работу с холодным стартом на будущее** (из лекция https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla)

* Рекомендовать популярные
* Построить модель на соцдеме или других данных (Например, женщинам рекомендовать женскую одежду, мужчинам мужскую и тд). То есть построить набор популярных товаров, характерных для этого соцдема.
* Провести онбординг (некоторые сервисы при регистрации спрашивают, укажите что вам нравится)

### Стратегия работы с пропусками

* Удалять нельзя.
* На данном этапе оставить без изменения.
* На этапе моделирования заменить все NaN на специальный обучаемый неизвестный эмбеддинг, чтобы дать модели самой выучить специальный, 
единый эмбеддинг для всех "неизвестных" брендов.
* Со звездочкой. Использовать гибридный подход: Для "холодных" брендов можно пытаться сгенерировать временный эмбеддинг 
на основе их других характеристик (категория, цена, текстовое описание, если оно есть). Это называется гибридной рекомендательной 
системой, которая смешивает коллаборативную фильтрацию с подходом на основе контента

# Заполняем пропуски, чтобы потом посмотреть основные статистики

## Обработка users_df (совпадает во всех доменах)
В таблице users_df заполняем пропуски в закодированных категориальных признаках region и socdem_cluster числом -1. Мы это можем сделать, так как минимальные значения этих признаков равны нулю.

In [50]:
users_df['region'] = users_df['region'].fillna(-1)
users_df['socdem_cluster'] = users_df['socdem_cluster'].fillna(-1)

In [51]:
# Меняем тип данных на целочисленный
users_df['region'] = users_df['region'].astype(int)
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype(int)

In [52]:
print('Пропуски в users_df после обработки:')
print(users_df.isnull().sum())

Пропуски в users_df после обработки:
user_id           0
socdem_cluster    0
region            0
dtype: int64


## Обработка items_df

In [53]:
# Заполняем категориальные пропуски строкой 'unknown'
items_df['category'] = items_df['category'].fillna('unknown')
items_df['subcategory'] = items_df['subcategory'].fillna('unknown')

In [54]:
# Рассчитываем медианную цену
median_price = items_df['price'].median()
print(f'Медианная цена для заполнения пропусков: {median_price}')

Медианная цена для заполнения пропусков: -3.875121896478133


In [55]:
items_df['price'].min()

np.float64(-10.0)

В признаке price присутствуют отрицательные значения. Это не выбросы. В описании дата сета price - представляет собой денежную стоимость взаимодействия. Цены в каталогах переведены в промежуток от −10 до 10. 

In [56]:
# Заполняем пропуски в цене медианой
items_df['price'] = items_df['price'].fillna(median_price)

In [57]:
print("Пропуски в items_df после обработки:")
print(items_df.isnull().sum())

Пропуски в items_df после обработки:
item_id        0
brand_id       0
category       0
subcategory    0
price          0
embedding      0
dtype: int64


## Обработка brands_df

Пропуски в embedding не трогаем. Они важны как сигнал о "холодных" брендах.

# Основные статистики

### Анализ таблицы users_df (совпадает во всех доменах)

Признаки 'socdem_cluster', 'region' это категориальные, закодированные числами, метод describe тут не поможет.

In [58]:
print(f'Уникальных пользователей: {users_df['user_id'].nunique()}')
print(f'Уникальных регионов: {users_df['region'].nunique()}')
print(f'Уникальных социо-демографических кластеров: {users_df['socdem_cluster'].nunique()}')

Уникальных пользователей: 3500000
Уникальных регионов: 91
Уникальных социо-демографических кластеров: 22


In [59]:
# Посмотрим, сколько пользователей в каждом регионе
users_df['region'].value_counts()

region
2     551358
60    249920
90    181623
18    137250
24    114992
       ...  
65      1687
30      1125
69       915
39       836
20       338
Name: count, Length: 91, dtype: int64

In [60]:
# Посмотрим, сколько пользователей в каждом кластере
users_df['socdem_cluster'].value_counts()

socdem_cluster
 17    473629
 20    420944
 12    416287
 9     340144
 21    331140
 10    266045
 19    261186
 4     254800
 0     205299
 5     148037
 7     140750
 18    105353
 13     40572
 6      36130
 1      18629
 2      12134
 16      9859
 8       5825
-1       5153
 3       3900
 11      2637
 15      1547
Name: count, dtype: int64

In [61]:
# Преобразуем столбцы в категориальный тип
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype('category')
users_df['region'] = users_df['region'].astype('category')

In [62]:
# И применим describe
print(users_df[['socdem_cluster', 'region']].describe())

        socdem_cluster   region
count          3500000  3500000
unique              22       91
top                 17        2
freq            473629   551358


### Анализ items_df

In [63]:
desc = items_df['price'].describe()
print(desc.apply(lambda x: f'{x:.5f}'))

count    250171.00000
mean         -3.77987
std           1.25295
min         -10.00000
25%          -4.69678
50%          -3.87512
75%          -3.05238
max           7.74009
Name: price, dtype: object


* Из таблицы видно, что среднее значение и медиана очень близки друг к другу, распределение практически симметричное.
* Цены сосредоточены в диапазоне -4.7 до -3.05
* Максимальное значение 7.74 очень сильно отличается от 75-го процентиля. Это явный признак наличия выбросов. Наличие товаров, которые значительно дороже основной массы товаров.

Так как Retail это поведение в сервисе доставки продуктов: от просмотра до оформления заказа. То возможно, что сильно дорогие товары это срочные/экспресс доставки (если в стоимость айтема закладывается тариф доставки).

### Анализ events_df

In [64]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200006078,8923243,fmcg_208146,catalog,view,android
1,113011200032272,12124655,fmcg_1034096,catalog,view,android
2,113011200077336,20658160,fmcg_112697,catalog,view,android
3,113011200112253,85296647,fmcg_618508,catalog,view,android
4,113011200250868,85296647,fmcg_509248,catalog,view,android


In [65]:
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

#### Определение самых популярных товаров

События типа 'view' фиксируют только факт показа товара пользователю, и многие из них могут быть случайными или малозначимыми (пользователь просто пролистал страницу, товар попал в рекомендацию, но не заинтересовал). Поэтому топ по просмотрам покажет товары, которые чаще всего показывались, но не обязательно были востребованы.

**Важное из лекции** https://vkvideo.ru/video-123851409_456239385?list=ln-fi0AbfTmgwRkUuI0Ai :

* Популярность не считается по просмотрам. Потому что просмотеры зависят от самой рекомендательной системы, и не зависят от дейтсвий пользователя.

Для выявления действительно популярных и интересных товаров лучше использовать более сильные сигналы, такие как:

'click' — пользователь проявил явный интерес, кликнув на товар.

'like' — проявил позитивную реакцию, добавил в избранное.

'clickout' — намерение к покупке, переход к оформлению или внешнему сайту.

'added-to-cart' - добавление в корзину

In [66]:
# Вспомним, какие есть типы взаимодействий
recent_events['action_type'].unique()

array(['view', 'click', 'added-to-cart'], dtype=object)

In [67]:
# Берем только клики и считаем количества кликов для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'click']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по кликам
print(top_popular_items.head(10))

item_id
fmcg_514296    495
fmcg_651828    431
fmcg_346278    349
fmcg_160165    346
fmcg_450490    313
fmcg_671193    305
fmcg_722287    296
fmcg_292617    290
fmcg_335947    252
fmcg_250877    242
Name: count, dtype: int64


In [68]:
# Берем только добавление в корзину
top_popular_items = recent_events[recent_events['action_type'] == 'added-to-cart']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по лайкам
print(top_popular_items.head(10))

item_id
fmcg_651828    2417
fmcg_514296    2246
fmcg_450490    1577
fmcg_292617    1485
fmcg_722287    1360
fmcg_250877    1328
fmcg_335947    1280
fmcg_671193    1259
fmcg_160165    1125
fmcg_510033    1066
Name: count, dtype: int64


Создадим столбец label. Действию view в колонке label поставим в соответсвие 0, а действиям click и add-to-card поставим в соответсвие 1.

In [69]:
recent_events['label'] = recent_events['action_type'].map({'view':0, 'click':1, 'added-to-cart':1})

In [70]:
recent_events.groupby('action_type')['label'].mean()

action_type
added-to-cart    1.0
click            1.0
view             0.0
Name: label, dtype: float64

In [71]:
# Популярность по количеству уникальных пользователей. 
# Именно такой способ расчета популярности часто используется в рекомендательных системах

# Считаем количество уникальных пользователей, совершивших значимые действия для каждого товара
label_1_recent_events = recent_events[recent_events['label'] == 1]
unique_user_popularity = label_1_recent_events.groupby('item_id')['user_id'].nunique().sort_values(ascending=False)

print(unique_user_popularity.head(10))

item_id
fmcg_651828    1988
fmcg_514296    1854
fmcg_450490    1379
fmcg_292617    1328
fmcg_250877    1223
fmcg_722287    1203
fmcg_671193    1131
fmcg_335947    1127
fmcg_160165    1049
fmcg_510033     943
Name: user_id, dtype: int64


# Анализ взаимосвязи переменных

В items_df содержится больше всего признаков, которые могут быть взаимосвязаны. Добавим к ним признаки из events. И все категориальные закодируем числами.

In [72]:
# Нам нужны признаки в числовом формате
# Кодирование категориальных признаков
# Метод .cat.codes присваивает уникальный числовой код каждой категории.

Важно понимать, что .cat.codes можно использовать для анализа корреляций. Но нельзя использовать для линейных моделей. Для линейных моделей лучше использовать one-hot encoding

In [73]:
recent_events.columns

Index(['timestamp', 'user_id', 'item_id', 'subdomain', 'action_type', 'os',
       'label'],
      dtype='object')

In [74]:
items_df.columns

Index(['item_id', 'brand_id', 'category', 'subcategory', 'price', 'embedding'], dtype='object')

In [75]:
# Объединяем датафреймы items_df и recent_events
item_features = items_df[['item_id', 'price', 'brand_id', 'category', 'subcategory']]
merged_df = pd.merge(recent_events, item_features, on='item_id', how='left')

In [76]:
merged_df.columns

Index(['timestamp', 'user_id', 'item_id', 'subdomain', 'action_type', 'os',
       'label', 'price', 'brand_id', 'category', 'subcategory'],
      dtype='object')

In [77]:
# Список всех категориальных колонок, которые мы хотим проанализировать
categorical_features = ['brand_id', 'category', 'subcategory', 'subdomain', 'os']

In [78]:
analysis_df = merged_df.copy()

In [79]:
#Кодируем категории
for feature in categorical_features:
    analysis_df[f'{feature}_encoded'] = analysis_df[feature].astype('category').cat.codes

In [80]:
# Выбираем столбцы для корреляционной матрицы: цена + все закодированные признаки
features_for_heatmap = ['price'] + [f'{feature}_encoded' for feature in categorical_features]

In [81]:
correlation_matrix_full = analysis_df[features_for_heatmap].corr()
correlation_matrix_full

,price,brand_id_encoded,category_encoded,subcategory_encoded,subdomain_encoded,os_encoded
price,1.000000,0.180849,-0.026979,0.085584,-0.017415,0.013787
brand_id_encoded,0.180849,1.000000,-0.024669,-0.049996,0.021530,0.000422
category_encoded,-0.026979,-0.024669,1.000000,0.299942,-0.010391,-0.007044
subcategory_encoded,0.085584,-0.049996,0.299942,1.000000,-0.056739,-0.017197
subdomain_encoded,-0.017415,0.021530,-0.010391,-0.056739,1.000000,0.244524
os_encoded,0.013787,0.000422,-0.007044,-0.017197,0.244524,1.000000


* Большинство взаимосвязей довольно слабые, что снижает риски мультиколлинеарности при построении моделей
* Категория и подкатегория имеют заметную положительную корреляцию (0.3) - это логично, так как подкатегории вложены в категории
* Бренд является наиболее значимым фактором для цены среди рассмотренных признаков
* Операционная система и поддомен (место взаимодействия) имеют заметную корреляцию (0.25). Пользователи на разных операционных системах имеют разные паттерны взаимодействия с платформой. Например, может быть так, что пользователи чаще используют поиск на одной ос, а на другой - рекомендации.

# Целевая переменная

На основе семинара-лекции: https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla


* У нас есть список релевантных пользователю товаров в обучающей выборке (то есть  айдишники товаров, с которым пользователь сделал значимые интеракции на обучающем промежутке времени). И мы хотим предсказать айдишники товаров, с которыми пользователь провзаимодейтсвует на тестовом (отложенном) периоде времени. Для подсчета метрик у нас должны быть ground truth айтемы, про которые мы знаем, что они релевантны пользователю и мы их будем сравнивать с предсказанными. Ground truth айтемы в рекомендательных системах из данного датасета обычно выбираются как реальные положительные взаимодействия пользователя с айтемами в тестовом временном окне.


* Целевая переменная (таргет) — это список или множество айтемов, с которыми пользователь реально взаимодействовал в тестовом (отложенном) временном промежутке.

* Таргет формируется из положительных (значимых) взаимодействий пользователя с айтемами на отложенном периоде, например, кликов, переходов к покупке, лайков и т.п.

* Эти реальные положительные взаимодействия и есть ground truth айтемы, с которыми сравниваются предсказания модели.

* В обучающей выборке у нас есть история взаимодействий пользователя с релевантными товарами за период до теста - на её основе строится модель.

* В тесте модель должна предсказать айтемы, которые user выберет/кликнет/купит. Это и есть задача.

Иными словами, таргет это бинарная метка релевантности айтема пользователю в тестовом периоде, выраженная в виде множества ground truth айтемов. 